In [103]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix

In [104]:
df=pd.read_csv(r'C:\Users\USER\Desktop\5th sem\project1_daibatise\data\processed\mohammed_ajlan_p1w1_features.csv')

In [105]:
feature_cols = ["age","pregnancies","glucose","blood_pressure",
                "skin_thickness","insulin","bmi","diabetes_pedigree",
                "bmi_category","glucose_band"]
x = df[feature_cols].copy()
y = df["outcome"]
x

,age,pregnancies,glucose,blood_pressure,skin_thickness,insulin,bmi,diabetes_pedigree,bmi_category,glucose_band
0,79,0,124.000000,71.0,30.0,124.0,23.4,0.588,normal,prediabetic
1,37,0,153.000000,85.0,24.0,42.0,33.9,0.192,obese,diabetic
2,39,0,142.000000,68.0,22.0,159.0,26.9,0.777,overweight,diabetic
3,68,7,121.000000,88.0,26.0,124.0,29.0,1.217,overweight,prediabetic
4,75,0,107.000000,84.0,31.0,101.0,21.2,0.278,normal,prediabetic
...,...,...,...,...,...,...,...,...,...,...
945,64,2,117.855165,81.0,34.0,124.0,40.3,0.641,obese,prediabetic
946,71,0,140.000000,64.0,37.0,124.0,33.4,0.116,obese,diabetic
947,40,0,130.000000,73.0,26.0,258.0,20.1,0.709,normal,diabetic
948,52,4,123.000000,76.0,22.0,189.0,38.9,0.540,obese,prediabetic


In [106]:
x=pd.get_dummies(x,drop_first=True)
x.shape

(950, 13)

In [107]:
x_train,x_test,y_train,y_test= train_test_split(
    x,y,test_size=0.2,random_state=42,stratify=y)

In [108]:
scaler = StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)#Calculates the mean and standard deviation from x train
x_test_scaled=scaler.transform(x_test)#Uses those values to standardize x_train.

In [109]:
logreg = LogisticRegression(max_iter=1000).fit(x_train_scaled, y_train)
dtree = DecisionTreeClassifier(max_depth=3,random_state=42).fit(x_train, y_train)
knn = KNeighborsClassifier(n_neighbors=25).fit(x_train_scaled, y_train)



In [110]:
print('LogisticRegression',accuracy_score(y_test,logreg.predict(x_test_scaled)))
print('DecisionTreeClassifier',accuracy_score(y_test,dtree.predict(x_test)))
print('kneighborsclassifier',accuracy_score(y_test,knn.predict(x_test_scaled)))

LogisticRegression 0.7526315789473684
DecisionTreeClassifier 0.7210526315789474
kneighborsclassifier 0.7473684210526316


In [111]:
models = [
    ("Logistic Regression", logreg, x_test_scaled),
    ("Decision Tree (d=3)", dtree, x_test),
    ("KNN (k=25)", knn, x_test_scaled)
]

print("\nTest set:", len(y_test), "patients |",
      int(np.sum(y_test)), "of them diabetic")


Test set: 190 patients | 55 of them diabetic


build consfuion matrix for every model

In [112]:
rows = []

for name, m, Xt in models:
    cm = confusion_matrix(y_test, m.predict(Xt))
    tn, fp, fn, tp = cm.ravel()

    rows.append({
        "model": name,
        "correct_negatives": int(tn),
        "false_alarms": int(fp),
        "patients_missed": int(fn),
        "patients_found": int(tp),
        "accuracy": round(accuracy_score(y_test, m.predict(Xt)), 4)
    })

matrices = pd.DataFrame(rows)
print(matrices.to_string(index=False))

              model  correct_negatives  false_alarms  patients_missed  patients_found  accuracy
Logistic Regression                127             8               39              16    0.7526
Decision Tree (d=3)                121            14               39              16    0.7211
         KNN (k=25)                132             3               45              10    0.7474


* **127 (TN):** 127 people were correctly identified as not having diabetes.
* **8 (FP):** 8 people were incorrectly identified as having diabetes.
* **39 (FN):** 39 people who actually had diabetes were missed by the model.
* **16 (TP):** 16 people who had diabetes were correctly identified by the model.

* **Logistic Regression:** The main error is **missed patients (FN = 39)**.
* **Decision Tree:** The main error is **missed patients (FN = 39)**, along with 14 false alarms.
* **KNN:** The main error is **missed patients (FN = 45)**, which is the highest among the three models.


score the model that never says yes

In [113]:
from sklearn.dummy import DummyClassifier
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(x_train, y_train)
y_pred=dummy.predict(x_test)
confusion_matrix(y_test,y_pred)

array([[135,   0],
       [ 55,   0]])

In [114]:
print('DummyClassifier',accuracy_score(y_test,y_pred))

DummyClassifier 0.7105263157894737


In [115]:
y_test.value_counts(normalize=True)

outcome
0    0.710526
1    0.289474
Name: proportion, dtype: float64

# precision and recall

recall

Out of all the actual positive cases, how many did the model correctly identify?

Recall = True Positives (TP) / (True Positives (TP) + False Negatives (FN))

In [116]:
cm =confusion_matrix(y_test,logreg.predict(x_test_scaled))
tn,fp,fn,tp=cm.ravel()
print("recall:tp/tp+fn=",(tp/(tp+fn)))

recall:tp/tp+fn= 0.2909090909090909


In [117]:
from sklearn.metrics import recall_score

In [118]:
recall_score(y_test,logreg.predict(x_test_scaled))
print("DecisionTreeClassifier  Recall =",recall_score(y_test,dtree.predict(x_test)))
print("KNeighborsClassifier  Recall =",recall_score(y_test,knn.predict(x_test_scaled)))
print("LogisticRegression Recall=",recall_score(y_test,logreg.predict(x_test_scaled)))

DecisionTreeClassifier  Recall = 0.2909090909090909
KNeighborsClassifier  Recall = 0.18181818181818182
LogisticRegression Recall= 0.2909090909090909


precision


In [119]:
cm =confusion_matrix(y_test,logreg.predict(x_test_scaled))
tn,fp,fn,tp=cm.ravel()
print("precision:tp/tp+fp=",(tp/(tp+fp)))

precision:tp/tp+fp= 0.6666666666666666


Precision tells us: Out of all the cases the model predicted as positive, how many were actually positive?

Precision = True Positives (TP) / (TP + False Positives (FP))

In [120]:
from sklearn.metrics import precision_score

In [121]:
precision_score(y_test,logreg.predict(x_test_scaled))
print("DecisionTreeClassifier  precision =",precision_score(y_test,dtree.predict(x_test)))
print("KNeighborsClassifier  precision =",precision_score(y_test,knn.predict(x_test_scaled)))
print("LogisticRegression precision=",precision_score(y_test,logreg.predict(x_test_scaled)))

DecisionTreeClassifier  precision = 0.5333333333333333
KNeighborsClassifier  precision = 0.7692307692307693
LogisticRegression precision= 0.6666666666666666


In [122]:
rows=[]
for name,m,xt in models:
    pred = m.predict(xt)
    cm=confusion_matrix(y_test,pred)
    tn,fp,fn,tp=cm.ravel()
    rows.append({
              "model":name,
              "recall":round(recall_score(y_test,pred,zero_division=0),4),
              "precision":round(precision_score(y_test,pred,zero_division=0),4),
              "accuracy":round(accuracy_score(y_test,pred),4),
              "correct_negatives":int(tn),"false_alarams":int(fp),
              "missed_patients":int(fn),"found_patients":int(tp)})

results=pd.DataFrame(rows)
results

,model,recall,precision,accuracy,correct_negatives,false_alarams,missed_patients,found_patients
0,Logistic Regression,0.2909,0.6667,0.7526,127,8,39,16
1,Decision Tree (d=3),0.2909,0.5333,0.7211,121,14,39,16
2,KNN (k=25),0.1818,0.7692,0.7474,132,3,45,10


by precision

In [123]:
results.sort_values('precision',ascending=False)

,model,recall,precision,accuracy,correct_negatives,false_alarams,missed_patients,found_patients
2,KNN (k=25),0.1818,0.7692,0.7474,132,3,45,10
0,Logistic Regression,0.2909,0.6667,0.7526,127,8,39,16
1,Decision Tree (d=3),0.2909,0.5333,0.7211,121,14,39,16


by recall

In [124]:
results.sort_values('recall',ascending=False)

,model,recall,precision,accuracy,correct_negatives,false_alarams,missed_patients,found_patients
0,Logistic Regression,0.2909,0.6667,0.7526,127,8,39,16
1,Decision Tree (d=3),0.2909,0.5333,0.7211,121,14,39,16
2,KNN (k=25),0.1818,0.7692,0.7474,132,3,45,10


In [125]:
probs=logreg.predict_proba(x_test_scaled)[:,1]
preds=logreg.predict(x_test_scaled)
pd.DataFrame({"probability":probs[:10].round(3),
              "prediction":preds[:10],
              "actual": np.asarray(y_test)[:10]

})

,probability,prediction,actual
0,0.289,0,0
1,0.032,0,0
2,0.182,0,0
3,0.182,0,0
4,0.177,0,1
5,0.161,0,0
6,0.419,0,0
7,0.473,0,1
8,0.286,0,0
9,0.121,0,0


In [126]:
sweep=[]
for t in[0.5,0.45,0.4,0.35,0.3,0.25,0.2,0.15]:
    pred_t=(probs>=t).astype(int)
    cm=confusion_matrix(y_test,pred_t)
    tn,fp,fn,tp=cm.ravel()
    sweep.append({
              "threshold":t,
              "recall":round(recall_score(y_test,pred_t,zero_division=0),4),
              "precision":round(precision_score(y_test,pred_t,zero_division=0),4),
              "accuracy":round(accuracy_score(y_test,pred_t),4),
              "correct_negatives":int(tn),"false_alarams":int(fp),
              "missed_patients":int(fn),"found_patients":int(tp)})

sweep=pd.DataFrame(sweep)
sweep

,threshold,recall,precision,accuracy,correct_negatives,false_alarams,missed_patients,found_patients
0,0.50,0.2909,0.6667,0.7526,127,8,39,16
1,0.45,0.3636,0.5556,0.7316,119,16,35,20
2,0.40,0.4364,0.5000,0.7105,111,24,31,24
3,0.35,0.5636,0.5082,0.7158,105,30,24,31
4,0.30,0.6182,0.4789,0.6947,98,37,21,34
5,0.25,0.6909,0.4043,0.6158,79,56,17,38
6,0.20,0.7818,0.3874,0.5789,67,68,12,43
7,0.15,0.8727,0.3692,0.5316,53,82,7,48


# F1  SCORE

F1 score is a measure that combines Precision and Recall to show how well a classification model performs.

F1=2×Precision+RecallPrecision×Recall​

In [127]:
from sklearn.metrics import f1_score

In [128]:
print('LogisticRegression',f1_score(y_test,logreg.predict(x_test_scaled)))
print('DecisionTreeClassifier',f1_score(y_test,dtree.predict(x_test)))
print('kneighborsclassifier',f1_score(y_test,knn.predict(x_test_scaled)))

LogisticRegression 0.4050632911392405
DecisionTreeClassifier 0.3764705882352941
kneighborsclassifier 0.29411764705882354


In [129]:
from sklearn.metrics import f1_score,classification_report
f1_score(y_test,pred)
print(classification_report(y_test,pred,digits=4))

              precision    recall  f1-score   support

           0     0.7458    0.9778    0.8462       135
           1     0.7692    0.1818    0.2941        55

    accuracy                         0.7474       190
   macro avg     0.7575    0.5798    0.5701       190
weighted avg     0.7526    0.7474    0.6864       190



In [130]:
results

,model,recall,precision,accuracy,correct_negatives,false_alarams,missed_patients,found_patients
0,Logistic Regression,0.2909,0.6667,0.7526,127,8,39,16
1,Decision Tree (d=3),0.2909,0.5333,0.7211,121,14,39,16
2,KNN (k=25),0.1818,0.7692,0.7474,132,3,45,10


In [136]:
results['f1']= [f1_score(y_test,m.predict(xt))
                for name,m,xt in models]

print(
    results[["model","recall","precision","accuracy","f1"]]
 
    .round(4)
)


                 model  recall  precision  accuracy      f1
0  Logistic Regression  0.2909     0.6667    0.7526  0.4051
1  Decision Tree (d=3)  0.2909     0.5333    0.7211  0.3765
2           KNN (k=25)  0.1818     0.7692    0.7474  0.2941


In [138]:

print("Dummy Accuracy:", accuracy_score(y_test, y_pred))
print("Dummy Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Dummy Recall:", recall_score(y_test, y_pred, zero_division=0))
print("Dummy F1:", f1_score(y_test, y_pred, zero_division=0))

Dummy Accuracy: 0.7105263157894737
Dummy Precision: 0.0
Dummy Recall: 0.0
Dummy F1: 0.0


find the thresold that  maximises f1

In [150]:
sweep['f1'] = [f1_score(y_test, (probs >= t).astype(int), zero_division=0)
               for t in sweep['threshold']]
print(sweep[['threshold', 'accuracy', 'precision', 'recall', 'f1','found_patients','missed_patients']]
.sort_values("f1", ascending=False).to_string(index=False))
best = sweep.loc[sweep["f1"].idxmax()]



 threshold  accuracy  precision  recall       f1  found_patients  missed_patients
      0.30    0.6947     0.4789  0.6182 0.539683              34               21
      0.35    0.7158     0.5082  0.5636 0.534483              31               24
      0.15    0.5316     0.3692  0.8727 0.518919              48                7
      0.20    0.5789     0.3874  0.7818 0.518072              43               12
      0.25    0.6158     0.4043  0.6909 0.510067              38               17
      0.40    0.7105     0.5000  0.4364 0.466019              24               31
      0.45    0.7316     0.5556  0.3636 0.439560              20               35
      0.50    0.7526     0.6667  0.2909 0.405063              16               39


compare the tuned model with the untuned one

In [153]:

tuned_pred = (probs >= 0.30).astype(int)

def score_row(label, pred):
    cm = confusion_matrix(y_test, pred)
    tn, fp, fn, tp = cm.ravel()

    return {
        "model": label,
        "accuracy": round(accuracy_score(y_test, pred), 4),
        "precision": round(precision_score(y_test, pred, zero_division=0), 4),
        "recall": round(recall_score(y_test, pred, zero_division=0), 4),
        "f1": round(f1_score(y_test, pred, zero_division=0), 4),
        "found": int(tp),
        "missed": int(fn),
        "false_alarms": int(fp)
    }

pd.DataFrame([
    score_row("Logistic Regression (threshold 0.50)", preds),
    score_row("Logistic Regression (threshold 0.30)", tuned_pred)
])

,model,accuracy,precision,recall,f1,found,missed,false_alarms
0,Logistic Regression (threshold 0.50),0.7526,0.6667,0.2909,0.4051,16,39,8
1,Logistic Regression (threshold 0.30),0.6947,0.4789,0.6182,0.5397,34,21,37
